# Notebook 02 — Probabilidad e inferencia estadística | A/B Testing en AWS

**Objetivo:** explicar la incertidumbre de las tasas de conversión del experimento A/B mediante probabilidad, simulación, error estándar e intervalos de confianza.  
**Entrada:** `s3://TU_BUCKET/processed/ab_data_clean.csv` generado en Notebook 01 (o ruta local).  
**Salidas:** CSV de tasas e intervalos, gráficas PNG y archivo de simulaciones resumidas en `results/`.

> **Alcance:** análisis exploratorio e inferencial de cada grupo por separado. La **prueba formal de hipótesis A vs. B** se reserva para el Notebook 03. Los IC suponen observaciones independientes y muestran incertidumbre de muestreo, no sesgos de selección. El conjunto limpio del Notebook 01 es una muestra de análisis simplificada (*per-protocol*), no necesariamente un análisis *intention-to-treat*.

## 1. Configurar el entorno
Usa Python 3 en SageMaker Studio JupyterLab. Ajusta `BUCKET` y `USE_S3`. Si trabajas sin S3, coloca el CSV de Notebook 01 en `data/ab_data_clean.csv`.

In [ ]:
# Si faltan paquetes, ejecuta una vez:
# %pip install pandas numpy matplotlib seaborn scipy statsmodels boto3
from pathlib import Path
from io import BytesIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import binom, norm, skew
from statsmodels.stats.proportion import proportion_confint
from IPython.display import display

SEED = 42
rng = np.random.default_rng(SEED)
ALPHA = 0.05
BUCKET = "TU_BUCKET"  # Cambiar
USE_S3 = True  # False para datos locales
INPUT_KEY = "processed/ab_data_clean.csv"
LOCAL_PATH = Path("data/ab_data_clean.csv")
OUTPUT_DIR = Path("outputs_notebook_02")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
sns.set_theme(style="whitegrid")
print("Entorno listo")

## 2. Cargar y validar datos
Cada `user_id` debe ser único y `converted` debe tomar 0 o 1. Estos controles evitan interpretar muestras erróneas como ensayos independientes.

In [ ]:
if USE_S3:
    if BUCKET == "TU_BUCKET":
        raise ValueError("Reemplaza TU_BUCKET por el nombre real de tu bucket S3")
    import boto3
    s3 = boto3.client("s3")
    response = s3.get_object(Bucket=BUCKET, Key=INPUT_KEY)
    df = pd.read_csv(BytesIO(response["Body"].read()))
else:
    df = pd.read_csv(LOCAL_PATH)

required = {"user_id", "group", "landing_page", "converted"}
assert required.issubset(df.columns), f"Columnas faltantes: {required - set(df.columns)}"
assert df["user_id"].is_unique, "Hay usuarios repetidos: revisar Notebook 01"
assert not df[list(required)].isna().any().any(), "Hay valores nulos en campos clave"
assert set(df["group"].unique()) == {"control", "treatment"}
assert df["converted"].isin([0, 1]).all()
assert ((df["group"].eq("control") & df["landing_page"].eq("old_page")) |
        (df["group"].eq("treatment") & df["landing_page"].eq("new_page"))).all()
print(f"Registros válidos: {len(df):,}")
display(df.head())

## 3. Estadística descriptiva: tasa de conversión
Una variable binaria se modela con una **Bernoulli**: $X\sim\mathrm{Bernoulli}(p)$, con $E[X]=p$ y $\operatorname{Var}(X)=p(1-p)$. La media muestral de `converted` es la tasa de conversión.

In [ ]:
summary = (df.groupby("group", observed=True)["converted"]
           .agg(usuarios="size", conversiones="sum", tasa="mean")
           .reindex(["control", "treatment"]))
summary["no_conversiones"] = summary["usuarios"] - summary["conversiones"]
summary["tasa_pct"] = 100 * summary["tasa"]
display(summary)
assert (summary["usuarios"] == summary["conversiones"]+summary["no_conversiones"]).all()

**Ejercicio 1.** Explica por qué `mean()` y `sum()/count()` producen la misma conversión cuando `converted` está codificada 0/1. ¿Qué ocurriría con valores faltantes?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(summary.index, summary["tasa_pct"])
ax.set(ylabel="Tasa de conversión (%)", xlabel="Grupo", title="Conversión observada por grupo")
ax.set_ylim(0, max(1, summary["tasa_pct"].max()*1.2))
for i, val in enumerate(summary["tasa_pct"]):
    ax.text(i, val + .15, f"{val:.3f}%", ha="center")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_conversion_observada.png", dpi=160)
plt.show()

## 4. Distribución binomial
Si hay $n$ usuarios con conversiones independientes y probabilidad común $p$, entonces $Y=\sum X_i\sim\operatorname{Binomial}(n,p)$. Su media es $np$ y su desviación estándar $\sqrt{np(1-p)}$. Usaremos parámetros **hipotéticos** para visualizar variabilidad; no son resultados del A/B.

In [ ]:
n_demo, p_demo, B = 1000, 0.12, 10000
sim_counts = rng.binomial(n=n_demo, p=p_demo, size=B)
print("Esperanza teórica:", n_demo*p_demo)
print("Desv. estándar teórica:", np.sqrt(n_demo*p_demo*(1-p_demo)))
print("Media simulada:", sim_counts.mean())
print("Desv. estándar simulada:", sim_counts.std(ddof=1))
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(sim_counts, bins=35, edgecolor="white")
ax.axvline(n_demo*p_demo, linestyle="--", color="black", label="Media teórica")
ax.set(xlabel="Conversiones de 1000 usuarios", ylabel="Experimentos simulados", title="10,000 simulaciones binomiales")
ax.legend(); fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_simulacion_binomial.png", dpi=160)
plt.show()

**Ejercicio 2.** Cambia `p_demo` a 0.05 y luego a 0.5. ¿Cómo cambia la variabilidad? Explica por qué el término $p(1-p)$ alcanza su máximo en $p=0.5$.

## 5. Probabilidades exactas de la binomial
Ejemplo **hipotético**: con 100 visitantes y $p=0.12$, ¿cuál es la probabilidad de observar al menos 15 conversiones? La función de supervivencia `sf(14)` calcula $P(Y\geq15)$.

In [ ]:
n_ex, p_ex = 100, 0.12
prob_at_least_15 = binom.sf(14, n_ex, p_ex)
prob_exactly_12 = binom.pmf(12, n_ex, p_ex)
print(f"P(Y >= 15) = {prob_at_least_15:.4%}")
print(f"P(Y = 12) = {prob_exactly_12:.4%}")

## 6. Error estándar de la proporción
$SE(\hat p)\approx\sqrt{\hat p(1-\hat p)/n}$. El error estándar mide cuánto variaría la estimación si repitiéramos el muestreo bajo los mismos supuestos; **no** es el porcentaje de errores del modelo.

In [ ]:
summary["se"] = np.sqrt(summary["tasa"]*(1-summary["tasa"])/summary["usuarios"])
summary["se_pp"] = 100*summary["se"]
display(summary[["usuarios", "tasa_pct", "se", "se_pp"]])
print("Interpretación: SE está en unidades de proporción; SE_pp, en puntos porcentuales")

## 7. Teorema central del límite (TCL) mediante simulación
Tomamos como probabilidades generadoras las conversiones observadas, **solo con fines didácticos**. Repetimos muchos estudios simulados para entender cómo se distribuye una proporción muestral. Al aumentar $n$, se concentra alrededor de $p$.

In [ ]:
p_sim = float(summary.loc["control", "tasa"])
B = 5000
sizes = [100, 1000, 10000]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, sample_n in zip(axes, sizes):
    sample_rates = rng.binomial(sample_n, p_sim, size=B)/sample_n
    ax.hist(sample_rates, bins=35, edgecolor="white", density=True)
    ax.axvline(p_sim, color="black", linestyle="--")
    ax.set(title=f"n = {sample_n:,}", xlabel="Proporción simulada", ylabel="Densidad")
fig.suptitle("Distribución muestral de una proporción")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_tcl_tamanos_muestra.png", dpi=160)
plt.show()

**Ejercicio 3.** Justifica por qué la distribución de `converted` es binaria, pero la distribución de sus medias puede aproximarse a una normal. Comprueba $np$ y $n(1-p)$ antes de apoyarte en la aproximación.

## 8. Intervalos de confianza de cada tasa (Wilson)
El intervalo del 95% de Wilson ofrece mejor comportamiento que el intervalo Wald básico, especialmente para proporciones cercanas a 0 o 1. **Interpretación frecuentista:** si repitiéramos el procedimiento en muchas muestras, aproximadamente el 95% de intervalos construirían una cobertura del parámetro verdadero. No hay una probabilidad del 95% asociada a este intervalo fijo una vez observado.

In [ ]:
ci_rows = []
for group, r in summary.iterrows():
    lower, upper = proportion_confint(
        count=int(r["conversiones"]), nobs=int(r["usuarios"]),
        alpha=ALPHA, method="wilson")
    ci_rows.append({"group":group, "usuarios":int(r["usuarios"]),
                    "conversiones":int(r["conversiones"]),
                    "tasa":float(r["tasa"]), "se":float(r["se"]),
                    "ic95_inf":lower, "ic95_sup":upper})
ci_df = pd.DataFrame(ci_rows).set_index("group")
display(ci_df)
for group, r in ci_df.iterrows():
    print(f"{group}: tasa={r['tasa']:.3%} | IC 95% Wilson [{r['ic95_inf']:.3%}, {r['ic95_sup']:.3%}]")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(ci_df))
y = 100*ci_df["tasa"].to_numpy()
yerr = np.vstack([100*(ci_df["tasa"]-ci_df["ic95_inf"]).to_numpy(),
                  100*(ci_df["ic95_sup"]-ci_df["tasa"]).to_numpy()])
ax.errorbar(x, y, yerr=yerr, fmt="o", capsize=8, markersize=7)
ax.set_xticks(x, ci_df.index)
ax.set(ylabel="Conversión (%)", title="Tasa observada e IC de Wilson 95% por grupo")
ax.grid(axis="y", alpha=.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_intervalos_wilson.png", dpi=160)
plt.show()

> **Advertencia:** que dos intervalos individuales se superpongan o no **no** reemplaza una prueba apropiada ni un intervalo para $p_B-p_A$. En Notebook 03 estimaremos directamente la diferencia y haremos la prueba Z de dos proporciones.

## 9. Bootstrap de la tasa de control
Remuestreamos observaciones binarias del grupo control **con reemplazo**. Para acelerar el cálculo y ahorrar memoria, usar una binomial con $n$ y $\hat p$ es matemáticamente equivalente a remuestrear `n` observaciones de una muestra Bernoulli. Calcularemos un IC percentil para la tasa. **No es un intervalo de diferencia ni una prueba de hipótesis.**

In [ ]:
n_control = int(ci_df.loc["control", "usuarios"])
p_control = float(ci_df.loc["control", "tasa"])
B = 5000
boot_rates = rng.binomial(n_control, p_control, size=B)/n_control
boot_low, boot_high = np.quantile(boot_rates, [ALPHA/2, 1-ALPHA/2])
print(f"Bootstrap percentil (control): [{boot_low:.3%}, {boot_high:.3%}]")
print(f"Wilson (control): [{ci_df.loc['control', 'ic95_inf']:.3%}, {ci_df.loc['control', 'ic95_sup']:.3%}]")
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(100*boot_rates, bins=40, edgecolor="white")
for value in (boot_low, boot_high): ax.axvline(100*value, color="black", linestyle="--")
ax.set(xlabel="Conversión bootstrap (%)", ylabel="Remuestreos", title="Incertidumbre de la tasa de control")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "05_bootstrap_control.png", dpi=160)
plt.show()

**Ejercicio 4.** Compara el IC Wilson con el IC bootstrap del control. ¿Son iguales? ¿Qué ocurre al aumentar `B` a 20,000? ¿Qué cambia cuando reduces artificialmente `n_control` a 100 (en una simulación, sin modificar los datos originales)?

## 10. Riesgos de inferencia y control de calidad

- La independencia entre usuarios es un supuesto; múltiples exposiciones, bots y clustering pueden violarlo.
- El dataset limpio es *per-protocol*. Una decisión real de implementación requeriría analizar cómo se asignaron los usuarios, la exposición real y exclusiones.
- Los intervalos cuantifican incertidumbre **muestral**, no sesgos por selección, instrumentación o cambios temporales.
- No interpretar diferencias entre A/B todavía: falta la prueba de hipótesis formal y el intervalo de la **diferencia**.
- Antes del experimento debe predefinirse métrica primaria, horizonte, tamaño muestral, nivel $\alpha$ y MDE.

## 11. Exportar resultados a disco y S3
`results/notebook02_conversion_intervals.csv` conserva los resultados estadísticos calculados; las imágenes se guardan en `results/notebook02/`. No sobreescribimos el CSV limpio del Notebook 01.

In [ ]:
ci_export = ci_df.reset_index()
summary_file = OUTPUT_DIR / "notebook02_conversion_intervals.csv"
ci_export.to_csv(summary_file, index=False)
if USE_S3:
    import boto3
    s3 = boto3.client("s3")
    for path in OUTPUT_DIR.iterdir():
        if path.is_file():
            key = ("results/notebook02_conversion_intervals.csv" if path.name == summary_file.name
                   else f"results/notebook02/{path.name}")
            s3.upload_file(str(path), BUCKET, key)
            print(f"Guardado s3://{BUCKET}/{key}")
else:
    print(f"Resultados en {OUTPUT_DIR.resolve()}")

## 12. Actividad final y preparación del Notebook 03

Escribe en Markdown, con tus resultados reales:
1. ¿Cuáles son las conversiones y los IC 95% por grupo?
2. ¿Cómo cambia la incertidumbre al incrementar el tamaño de muestra?
3. ¿Qué diferencias observas entre Wilson y bootstrap?
4. ¿Qué limitaciones del experimento afectan la interpretación?

**En el Notebook 03:** hipótesis $H_0:p_B=p_A$, estadístico Z, p-value bilateral, IC de Newcombe para $p_B-p_A$, lift, potencia y recomendación de negocio.